Compiles BUSCO scores and gap counts from raw output files into data/assembly_qc_sorted.csv.
Input: BUSCO summary files, gap BED files, metadata
Output: data/assembly_qc_sorted.csv

In [8]:
import pandas as pd
import os

In [ ]:
metadata_df = pd.read_csv("data/metadata.tsv", sep="\t")

In [10]:
# gaps
folder = "data/gaps/"

gaps_list = []
for index, row in metadata_df.iterrows():
    species = row["Species"]
    # gaps data in data/gaps/{species}/{chromosome}.gaps.bed (number of lines = number of gaps)

    species_folder = os.path.join(folder, species)
    total_gaps = 0
    chr_count = 0
    for filename in os.listdir(species_folder):
        if filename.endswith(".gaps.bed"):
            chr_count += 1
            filepath = os.path.join(species_folder, filename)
            with open(filepath, "r") as f:
                lines = f.readlines()
                total_gaps += len(lines)
    avg_gaps_per_chr = total_gaps / chr_count if chr_count > 0 else 0
    avg_gaps_per_Mb = total_gaps / row["Assembly size (Mb)"] 
    gaps_list.append({
        "Species": species,
        "Total gaps": total_gaps,
        "Avg gaps per chromosome": avg_gaps_per_chr,
        "Avg gaps per Mb": avg_gaps_per_Mb
    })
gaps_df = pd.DataFrame(gaps_list)

In [11]:
import os
import pandas as pd
import re

# BUSCOs
folder = "data/busco/"
busco_list = []

for index, row in metadata_df.iterrows():
    species = row["Species"]
    filepath = os.path.join(folder, species, f"short_summary.specific.viridiplantae_odb10.{species}.txt") 
    
    busco_stats = {
        "Species": species,
        "Complete_BUSCOs": 0,
        "Single_copy_BUSCOs": 0,
        "Duplicated_BUSCOs": 0,
        "Fragmented_BUSCOs": 0,
        "Missing_BUSCOs": 0,
        "Total_BUSCOs": 0,
        "Complete_pct": 0.0,
        "Single_copy_pct": 0.0,
        "Duplicated_pct": 0.0,
        "Fragmented_pct": 0.0,
        "Missing_pct": 0.0
    }
    
    # If file does not exist, skip
    if not os.path.exists(filepath):
        print(f"Warning: BUSCO file not found for {species}")
        busco_list.append(busco_stats)
        continue
    
    with open(filepath, "r") as f:
        for line in f:
            line = line.strip()
            
            # Parse the summary line with percentages
            if line.startswith("C:"):
                # Extract percentages using regex
                match = re.search(r'C:([\d.]+)%\[S:([\d.]+)%,D:([\d.]+)%\],F:([\d.]+)%,M:([\d.]+)%,n:(\d+)', line)
                if match:
                    busco_stats["Complete_pct"] = float(match.group(1))
                    busco_stats["Single_copy_pct"] = float(match.group(2))
                    busco_stats["Duplicated_pct"] = float(match.group(3))
                    busco_stats["Fragmented_pct"] = float(match.group(4))
                    busco_stats["Missing_pct"] = float(match.group(5))
                    busco_stats["Total_BUSCOs"] = int(match.group(6))
            
            # Parse the count lines
            elif "Complete BUSCOs (C)" in line:
                busco_stats["Complete_BUSCOs"] = int(line.split()[0])
            elif "Complete and single-copy BUSCOs (S)" in line:
                busco_stats["Single_copy_BUSCOs"] = int(line.split()[0])
            elif "Complete and duplicated BUSCOs (D)" in line:
                busco_stats["Duplicated_BUSCOs"] = int(line.split()[0])
            elif "Fragmented BUSCOs (F)" in line:
                busco_stats["Fragmented_BUSCOs"] = int(line.split()[0])
            elif "Missing BUSCOs (M)" in line:
                busco_stats["Missing_BUSCOs"] = int(line.split()[0])
            elif "Total BUSCO groups searched" in line:
                busco_stats["Total_BUSCOs"] = int(line.split()[0])
    
    busco_list.append(busco_stats)

busco_df = pd.DataFrame(busco_list)

In [12]:
merged_df = pd.merge(gaps_df, busco_df, on="Species")

In [13]:
order = ['Cuscuta_campestris', 'Cuscuta_epithymum', 
         'Poa_supina', 'Poa_infirma', 'Poa_chaixii', 'Poa_annua', 
         'Hordeum_vulgare', 'Secale_cereale', 'Triticum_aestivum', 'Triticum_monococcum', 
         'Brachypodium_hybridum', 'Brachypodium_distachyon', 'Brachypodium_stacei', 
         'Oryza_grandiglumis', 'Oryza_punctata', 'Oryza_sativa', 'Oryza_rufipogon', 'Oryza_glaberrima', 'Oryza_barthii', 
         'Oryza_australiensis', 'Oryza_coarctata', 'Oryza_ridleyi', 'Oryza_longiglumis', 'Oryza_brachyantha', 'Oryza_meyeriana', 
         'Zoysia_sinica', 'Zoysia_japonica', 'Panicum_miliaceum',
           'Luzula_pallescens', 'Luzula_sylvatica', 'Juncus_squarrosus', 'Juncus_effusus', 
           'Cyperus_fuscus', 'Cyperus_iria', 'Schoenoplectus_tabernaemontani', 'Schoenoplectus_triqueter', 'Schoenoplectus_lacustris',
           'Bolboschoenus_planiculmis', 'Eleocharis_quinqueflora', 'Eleocharis_dulcis', 'Carex_riparia', 'Carex_rostrata', 'Carex_hirta', 
           'Carex_nigra', 'Carex_elata', 'Carex_littledalei', 'Carex_depauperata', 'Carex_sylvatica', 'Carex_acutiformis', 'Carex_caryophyllea',
             'Carex_pendula', 'Carex_extensa', 'Carex_distans', 'Carex_divulsa', 'Carex_arenaria', 'Carex_echinata', 'Carex_myosuroides', 
             'Carex_laevigata', 'Scirpus_sylvaticus', 'Eriophorum_angustifolium', 'Eriophorum_vaginatum', 'Trichophorum_cespitosum', 
             'Rhynchospora_pubera', 'Rhynchospora_breviuscula', 'Rhynchospora_tenuis', 'Chamaelirium_luteum', 'Chionographis_japonica']

In [14]:
merged_df = merged_df[["Species","Avg gaps per chromosome", "Avg gaps per Mb", "Complete_pct"]]

merged_df['Species'] = pd.Categorical(
    merged_df['Species'],
    categories=order,
    ordered=True
)
merged_df = merged_df.sort_values('Species')

merged_df.to_csv("data/assembly_qc_sorted.csv", index=False)